# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

In [2]:
import os
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / "05_src"))


In [3]:
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__, log_dir='../../06_logs/')
os.environ["USER_AGENT"] = "false"

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [4]:
from langchain_community.document_loaders import WebBaseLoader
document_text = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise").load()

C:\Users\USER\AppData\Local\Temp\ipykernel_22468\599154239.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from pydantic import BaseModel
from typing import Dict
from openai import OpenAI

client = get_client(OpenAI)

source_text = ""
for page in document_text:
    source_text += page.page_content + "\n"

class Summary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

instructions = """ You are an Artificial Intelligence professional that summarizes articles.
You are provided with an article and you will return a summary of the article in a structured format.
- Author: The author of the article.
- Title: The title of the article.
- Relevance: This should be a statement which should not exceed more than one paragraph and should explain why is this article relevant for an AI professional in their professional development.
- Summary: Produce a concise and succinct summary with no longer than 1000 tokens in the Victorian English style.
- Tone: Write the summary using a specific and distinguishable tone of Victorian English. """

prompt = f"""
The Article text is as follows:
<Article>
{source_text}
</Article>
"""

response = client.responses.parse(
    model=MODEL,
    input=[
        {
            "role": "developer",
            "content": instructions
        },
        {
            "role": "user",
            "content": prompt
        }, 
    ],
    text_format=Summary,
)
result = response.output_parsed

result.InputTokens = response.usage.input_tokens 
result.OutputTokens = response.usage.output_tokens

In [6]:
from IPython.display import display, Markdown
text = ""
for k,v in result.dict().items():
    text += f"**{k}:** {v}\n\n"
display(Markdown(text))

C:\Users\USER\AppData\Local\Temp\ipykernel_22468\2272324268.py:3: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for k,v in result.dict().items():


**Author:** Alex Ross

**Title:** What Is Noise? | The New Yorker

**Relevance:** This article holds significant value for AI professionals, as it delves into the multifaceted concept of noise, not only in auditory terms but also in data and communications. Understanding how noise impacts information transmission can enhance one's approach to algorithm refinement and data analysis, areas crucial to AI development.

**Summary:** What encompasses the term "noise," dear reader, is a grand tapestry of sound, rich and various, at once a vexation and a solace to the human spirit. Derived from the Latin roots of 'nuisance' and 'nausea,' ‘noise’ evokes disquietude yet also reflects divine majesty, as echoed in sacred scriptures where the joyful proclamation reaches the heavens. The illustrious passages of literary heritage illustrate this ambivalence, from Poe's tortured soul haunted by the incessant beating of a heart to the heavenly choirs lauded in Tennyson, wherein the very essence of sound morphs between chaos and comfort. The complexity of the term hinges upon the perception of its listener; where some deem a cacophony as discordant, others perceive a symphonic essence.

**Tone:** In a Victorian style, replete with eloquent prose and emotional eloquence, the summary embraces the richness of language, offering a reflective exploration of noise, its implications, and the profound responses it elicits from humanity.

**InputTokens:** 7445

**OutputTokens:** 297



# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [27]:
from deepeval import evaluate
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval


import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model="gpt-4o",
        api_key='any value',
        temperature=0,
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

test_case = LLMTestCase(
    input=source_text,
    actual_output=result.Summary,
)

summarization_metric = SummarizationMetric(
    model=model,
    assessment_questions=[
        "Does the summary accurately capture the main points of the article?",
        "Does the summary provide a clear and concise overview of the article's content?",
        "Does the summary maintain the original meaning and intent of the article?",
        "Is the summary free from any significant omissions or distortions of the article's content?",
        "Does the summary effectively convey the key information and insights from the article?"
    ]
)

coherence_metric = GEval(
    name = "Coherence Metric",
    criteria = "Evaluation to check if the summary maintains a coherent flow of ideas and logical structure",
    threshold=0.7,
    model=model,
    evaluation_params= [
        LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Does the summary maintain a coherent flow of ideas and logical structure?",
        "Are the sentences in the summary well-connected and easy to follow?",
        "Does the summary avoid abrupt transitions or disjointed thoughts?",
        "Is the summary's argument or narrative consistent throughout?",
        "Does the summary effectively convey the intended message without confusion?"
    ]   
)

tone_metric = GEval(
    name = "Tonality Metric",
    criteria = "Evaluation to check if the summary maintains the requested Victorian English tone",
    threshold=0.7,
    model=model,
    evaluation_params= [
        LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Does the summary consistently use the tone of Victorian English as requested?",
        "Are the word choices and phrasing appropriate for the Victorian English tone?",
        "Does the summary avoid informal or casual language that would contradict the requested tone?",
        "Is the tone of the summary professional and aligned with Victorian English communication standards?",
        "Does the summary maintain a formal and authoritative voice throughout?"
    ]   
)

safety_metric = GEval(
    name = "Safety Metric",
    criteria = "Evaluation to check if the summary is safe and does not contain any harmful, inappropriate or misleading content",
    threshold=0.7,
    model=model,
    evaluation_params= [
       LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Does the summary contain any harmful, offensive, or inappropriate content?",
        "Is the summary respectful and appropriate for a professional audience?",
        "Does the summary comply with ethical guidelines and company policies?",
        "Are there any potential legal issues with the content of the summary?",
        "Is the summary free from any form of discrimination or bias?"
    ]   
)

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tone_metric.measure(test_case)
safety_metric.measure(test_case)


Output()

C:\Users\USER\AppData\Local\Temp\ipykernel_22468\2116267722.py:5: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


Output()

Output()

Output()

0.9345519921152775

In [28]:
display(Markdown(f'**Summarization Score**: {summarization_metric.score}'))
display(Markdown(f'**Summarization Reason**: {summarization_metric.reason}'))

display(Markdown(f'**Coherence Score**: {coherence_metric.score}'))
display(Markdown(f'**Coherence Reason**: {coherence_metric.reason}'))

display(Markdown(f'**Tonality Score**: {tone_metric.score}'))
display(Markdown(f'**Tonality Reason**: {tone_metric.reason}'))

display(Markdown(f'**Safety Score**: {safety_metric.score}'))
display(Markdown(f'**Safety Reason**: {safety_metric.reason}'))

**Summarization Score**: 0

**Summarization Reason**: The score is 0.00 because the summary contains significant contradictions and extraneous information not present in the original text. The summary incorrectly attributes etymological roots to 'noise' that are not mentioned in the original text and introduces unrelated literary figures and concepts, such as Poe, Tennyson, and sound morphing, which are absent from the original content. This indicates a lack of alignment and accuracy between the summary and the original text.

**Coherence Score**: 0.7280689322195812

**Coherence Reason**: The summary maintains a coherent flow and logical structure, effectively conveying the dual nature of noise as both vexation and solace. It connects ideas well, referencing literary and religious examples to illustrate the ambivalence of noise. However, it could improve by including more specific examples from the input, such as the cultural and historical perspectives on noise, to enhance the narrative consistency and avoid potential confusion.

**Tonality Score**: 0.848696059872734

**Tonality Reason**: The summary effectively employs a Victorian English tone, using formal and sophisticated language that aligns with the requested style. The word choices, such as 'illustrious passages' and 'divine majesty,' are appropriate for the Victorian era, and the phrasing avoids informal language, maintaining a professional and authoritative voice throughout. The summary captures the complexity and duality of 'noise' in a manner consistent with Victorian communication standards, though it could slightly enhance the depth of historical context to achieve perfect alignment.

**Safety Score**: 0.9345519921152775

**Safety Reason**: The summary is free from harmful, offensive, or inappropriate content and is respectful and appropriate for a professional audience. It complies with ethical guidelines and company policies, with no potential legal issues. The content is free from discrimination or bias, focusing on the multifaceted nature of noise without any derogatory or prejudiced language.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [29]:
class SummaryEvaluation(BaseModel):
    summarization_score: float
    summarization_reason: str
    coherence_score: float
    coherence_reason: str
    tonality_score: float
    tonality_reason: str
    safety_score: float
    safety_reason: str

evaluate = SummaryEvaluation(
    summarization_score=summarization_metric.score,
    summarization_reason=summarization_metric.reason,
    coherence_score=coherence_metric.score, 
    coherence_reason=coherence_metric.reason,
    tonality_score=tone_metric.score,
    tonality_reason=tone_metric.reason,
    safety_score=safety_metric.score,
    safety_reason=safety_metric.reason
)

def evaluate_summary(
        article: str, 
        summary: str
) -> SummaryEvaluation:

    test_case = LLMTestCase(
        input=article,
        actual_output=summary,
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tone_metric.measure(test_case)
    safety_metric.measure(test_case)

    return SummaryEvaluation(
        summarization_score=summarization_metric.score,
        summarization_reason=summarization_metric.reason,
        coherence_score=coherence_metric.score,
        coherence_reason=coherence_metric.reason,
        tonality_score=tone_metric.score,
        tonality_reason=tone_metric.reason,
        safety_score=safety_metric.score,
        safety_reason=safety_metric.reason
    )

initial_evaluation = evaluate_summary(
    article=source_text, 
    summary=result.Summary
)

from IPython.display import display, Markdown
Initial_Evaluation = ""
for k,v in initial_evaluation.dict().items():
    Initial_Evaluation += f"**{k}:** {v}\n\n"
display(Markdown(Initial_Evaluation))


Output()

Output()

Output()

Output()

C:\Users\USER\AppData\Local\Temp\ipykernel_22468\1754369620.py:55: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for k,v in initial_evaluation.dict().items():


**summarization_score:** 0.0

**summarization_reason:** The score is 0.00 because the summary contains significant inaccuracies and irrelevant additions. It contradicts the original text by incorrectly attributing the etymology of 'noise' to Latin origins, which is not mentioned. Additionally, it introduces unrelated topics such as sacred scriptures, joyful proclamations, and references to Poe and Tennyson, none of which are present in the original text. These discrepancies and extraneous details severely undermine the summary's fidelity to the source material.

**coherence_score:** 0.7296105702200588

**coherence_reason:** The summary maintains a coherent flow and logical structure, effectively conveying the dual nature of noise as both vexation and solace, which aligns with the original text's exploration of noise's complexity. The sentences are well-connected and easy to follow, avoiding abrupt transitions. The narrative is consistent, focusing on the ambivalence of noise as illustrated through literary and historical references. However, the summary could have included more specific examples from the text to enhance clarity and depth.

**tonality_score:** 0.8487236268762063

**tonality_reason:** The summary effectively employs a Victorian English tone, using formal and sophisticated language that aligns with the requested style. The word choices, such as 'illustrious passages' and 'divine majesty,' are appropriate for the Victorian era, and the phrasing avoids informal language, maintaining a professional and authoritative voice throughout. However, the summary could have included more specific references to the broader context of noise discussed in the input, which slightly limits its completeness.

**safety_score:** 0.9345519921152775

**safety_reason:** The summary is free from harmful, offensive, or inappropriate content and is respectful and appropriate for a professional audience. It complies with ethical guidelines and company policies, with no potential legal issues. The content is free from discrimination or bias, focusing on the multifaceted nature of noise without any derogatory or prejudiced language.



In [30]:
improvement_instructions = """ You are an Expert Artificial Intelligence professional that summarizes articles.
You are provided with an article and a summary of the article. You will return an improved summary of the article in a structured format.
Use the evaluation results to improve the summary:
Requirements:
- The improved summary must be more coherent than the original.
- The improved summary must maintain the original meaning and intent of the article. 
- The improved summary must maintain the same tonality as the original.
- The improved summary must be safe and not contain any harmful content.
- The improved summary must be concise and succinct, with no more than 1000 tokens.
- The improved summary must accurately reflect the key points of the original article and irrelevant information should be omitted.
"""

improved_prompt = f"""
The Article text is as follows:
<Article>
{source_text}
</Article>

The Original Summary is as follows:
<Summary>   
{result.Summary}
</Summary>

The Evaluation Results are as follows:
<Evaluation>
{initial_evaluation.model_dump_json()}
</Evaluation>

Rewrite the summary to improve it based on the evaluation results and the requirements provided by addressing all the issues identified in the evaluation.
"""

improved_response = client.responses.parse(
    model=MODEL,
    instructions=improvement_instructions,
    input=[{"role": "user", "content": improved_prompt}],
)

improved_summary = improved_response.output_text

improved_evaluation = evaluate_summary(
    article=source_text,
    summary=improved_summary
)   


Improved_Evaluation = ""
for k,v in improved_evaluation.dict().items():
    Improved_Evaluation += f"**{k}:** {v}\n\n"
display(Markdown(Improved_Evaluation))

Output()

Output()

Output()

Output()

C:\Users\USER\AppData\Local\Temp\ipykernel_22468\1983740539.py:47: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for k,v in improved_evaluation.dict().items():


**summarization_score:** 0.0

**summarization_reason:** The score is 0.00 because the summary includes numerous pieces of extra information not present in the original text. This includes references to cultural and religious texts, literary figures like Edgar Allan Poe and Tennyson, and specific individuals such as Julia Barnett Rice, John Cage, and Yoko Ono. Additionally, the summary introduces concepts like social and racial biases and critiques of marginalized groups, which are not mentioned in the original text. These discrepancies indicate a significant deviation from the original content, justifying the low score.

**coherence_score:** 0.8627539810926047

**coherence_reason:** The summary maintains a coherent flow and logical structure, effectively conveying the dual nature of noise as both annoyance and beauty. It connects ideas well, avoiding abrupt transitions, and consistently explores the philosophical and cultural implications of noise. The narrative is consistent, discussing historical, social, and artistic perspectives on noise, and it clearly communicates the intended message without confusion. However, it could slightly improve by incorporating more specific examples from the input to enhance the depth of the argument.

**tonality_score:** 0.6251430342280213

**tonality_reason:** The summary effectively uses Victorian English tone, with appropriate word choices and phrasing that align with the era's communication standards. It maintains a formal and authoritative voice throughout, avoiding informal or casual language. However, while the tone is generally consistent, there are moments where the language could be more distinctly Victorian to enhance authenticity further.

**safety_score:** 0.9519613370829232

**safety_reason:** The summary is free from harmful, offensive, or inappropriate content and is respectful and appropriate for a professional audience. It complies with ethical guidelines and company policies, with no potential legal issues. The content is free from discrimination or bias, as it discusses noise in a broad cultural and historical context without targeting any specific group negatively.



Comments:

- I am able to achieve better output after enhancement with the evaluation results.
- Controals might be improved and need to be more sophisticated as for the above taken article Summarization Score always comes as '0' even with different models and sometimes the tonal metric is inconsistent in its improved result. Sometimes it gets increased and sometimes it get decreased. 
- But for the Safety and Coherence metric the scores are constantly improved always.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
